# Execution Pipeline

characters → [Tokenizer] → tokens → [Parser] → AST → [Compiler] → bytecode → [PVM] → behavior

In [1]:
price = 40
qty = 3

total = price * qty

print(total)

120


## Tokenizer: characters become words

In [13]:
import tokenize, io

code = b"price = 40\nqty = 3\ntotal = price * qty\nprint(total)"

for tok in tokenize.tokenize(io.BytesIO(code).readline):
    # print(f"Token Type: {tok.type}, Token Name: {tokenize.tok_name[tok.exact_type]}, Repr: {repr(tok.string)}")
    print(f"Full Token: {tok}")

Full Token: TokenInfo(type=68 (ENCODING), string='utf-8', start=(0, 0), end=(0, 0), line='')
Full Token: TokenInfo(type=1 (NAME), string='price', start=(1, 0), end=(1, 5), line='price = 40\n')
Full Token: TokenInfo(type=55 (OP), string='=', start=(1, 6), end=(1, 7), line='price = 40\n')
Full Token: TokenInfo(type=2 (NUMBER), string='40', start=(1, 8), end=(1, 10), line='price = 40\n')
Full Token: TokenInfo(type=4 (NEWLINE), string='\n', start=(1, 10), end=(1, 11), line='price = 40\n')
Full Token: TokenInfo(type=1 (NAME), string='qty', start=(2, 0), end=(2, 3), line='qty = 3\n')
Full Token: TokenInfo(type=55 (OP), string='=', start=(2, 4), end=(2, 5), line='qty = 3\n')
Full Token: TokenInfo(type=2 (NUMBER), string='3', start=(2, 6), end=(2, 7), line='qty = 3\n')
Full Token: TokenInfo(type=4 (NEWLINE), string='\n', start=(2, 7), end=(2, 8), line='qty = 3\n')
Full Token: TokenInfo(type=1 (NAME), string='total', start=(3, 0), end=(3, 5), line='total = price * qty\n')
Full Token: TokenInfo(

In [11]:
import tokenize, io

# using blank lines
code = b"price = 40\nqty = 3\n\ntotal = price * qty\n\nprint(total)"

for tok in tokenize.tokenize(io.BytesIO(code).readline):
    print(f"Token Type: {tok.type}, Token Name: {tokenize.tok_name[tok.exact_type]}, Repr: {repr(tok.string)}")

Token Type: 68, Token Name: ENCODING, Repr: 'utf-8'
Token Type: 1, Token Name: NAME, Repr: 'price'
Token Type: 55, Token Name: EQUAL, Repr: '='
Token Type: 2, Token Name: NUMBER, Repr: '40'
Token Type: 4, Token Name: NEWLINE, Repr: '\n'
Token Type: 1, Token Name: NAME, Repr: 'qty'
Token Type: 55, Token Name: EQUAL, Repr: '='
Token Type: 2, Token Name: NUMBER, Repr: '3'
Token Type: 4, Token Name: NEWLINE, Repr: '\n'
Token Type: 66, Token Name: NL, Repr: '\n'
Token Type: 1, Token Name: NAME, Repr: 'total'
Token Type: 55, Token Name: EQUAL, Repr: '='
Token Type: 1, Token Name: NAME, Repr: 'price'
Token Type: 55, Token Name: STAR, Repr: '*'
Token Type: 1, Token Name: NAME, Repr: 'qty'
Token Type: 4, Token Name: NEWLINE, Repr: '\n'
Token Type: 66, Token Name: NL, Repr: '\n'
Token Type: 1, Token Name: NAME, Repr: 'print'
Token Type: 55, Token Name: LPAR, Repr: '('
Token Type: 1, Token Name: NAME, Repr: 'total'
Token Type: 55, Token Name: RPAR, Repr: ')'
Token Type: 4, Token Name: NEWLINE, Re

In [12]:
# much formatted
import tokenize
import io

code = b"price = 40\nqty = 3\ntotal = price * qty\nprint(total)"

for tok in tokenize.tokenize(io.BytesIO(code).readline):
    print(f"{tok.string!r}")
    print(f"type       : {tok.type} ({tokenize.tok_name[tok.type]})")
    print(f"exact_type : {tok.exact_type} ({tokenize.tok_name[tok.exact_type]})")
    print()

'utf-8'
type       : 68 (ENCODING)
exact_type : 68 (ENCODING)

'price'
type       : 1 (NAME)
exact_type : 1 (NAME)

'='
type       : 55 (OP)
exact_type : 22 (EQUAL)

'40'
type       : 2 (NUMBER)
exact_type : 2 (NUMBER)

'\n'
type       : 4 (NEWLINE)
exact_type : 4 (NEWLINE)

'qty'
type       : 1 (NAME)
exact_type : 1 (NAME)

'='
type       : 55 (OP)
exact_type : 22 (EQUAL)

'3'
type       : 2 (NUMBER)
exact_type : 2 (NUMBER)

'\n'
type       : 4 (NEWLINE)
exact_type : 4 (NEWLINE)

'total'
type       : 1 (NAME)
exact_type : 1 (NAME)

'='
type       : 55 (OP)
exact_type : 22 (EQUAL)

'price'
type       : 1 (NAME)
exact_type : 1 (NAME)

'*'
type       : 55 (OP)
exact_type : 16 (STAR)

'qty'
type       : 1 (NAME)
exact_type : 1 (NAME)

'\n'
type       : 4 (NEWLINE)
exact_type : 4 (NEWLINE)

'print'
type       : 1 (NAME)
exact_type : 1 (NAME)

'('
type       : 55 (OP)
exact_type : 7 (LPAR)

'total'
type       : 1 (NAME)
exact_type : 1 (NAME)

')'
type       : 55 (OP)
exact_type : 8 (RPAR)



# AST

Parser doesn't directly execute but it builds AST.

In [1]:
import ast

code = """
price = 40
qty = 3
total = price * qty
print(total)
"""
print(ast.dump(ast.parse(code), indent=4))

Module(
    body=[
        Assign(
            targets=[
                Name(id='price', ctx=Store())],
            value=Constant(value=40)),
        Assign(
            targets=[
                Name(id='qty', ctx=Store())],
            value=Constant(value=3)),
        Assign(
            targets=[
                Name(id='total', ctx=Store())],
            value=BinOp(
                left=Name(id='price', ctx=Load()),
                op=Mult(),
                right=Name(id='qty', ctx=Load()))),
        Expr(
            value=Call(
                func=Name(id='print', ctx=Load()),
                args=[
                    Name(id='total', ctx=Load())]))])


# ByteCode
Bytecode is literally binary data, a sequence of 8-bit integers (bytes).

In [ ]:
# disassembler
import dis

def f():
    price = 40
    qty = 3
    total = price * qty
    print(total)

dis.dis(f)

  3           RESUME                   0

  4           LOAD_SMALL_INT          40
              STORE_FAST               0 (price)

  5           LOAD_SMALL_INT           3
              STORE_FAST               1 (qty)

  6           LOAD_FAST_BORROW_LOAD_FAST_BORROW 1 (price, qty)
              BINARY_OP                5 (*)
              STORE_FAST               2 (total)

  7           LOAD_GLOBAL              1 (print + NULL)
              LOAD_FAST_BORROW         2 (total)
              CALL                     1
              POP_TOP
              LOAD_CONST               1 (None)
              RETURN_VALUE


### Output

infix operation converted into postfix, i.e., price * qty -> price qty * 

In [3]:
import dis
dis.dis("total = price * qty")

  0           RESUME                   0

  1           LOAD_NAME                0 (price)
              LOAD_NAME                1 (qty)
              BINARY_OP                5 (*)
              STORE_NAME               2 (total)
              LOAD_CONST               0 (None)
              RETURN_VALUE


In [1]:
import dis

code = "price = 10\nqty = 20\ntotal = price * qty\nprint(total)"

dis.dis(code)

  0           RESUME                   0

  1           LOAD_SMALL_INT          10
              STORE_NAME               0 (price)

  2           LOAD_SMALL_INT          20
              STORE_NAME               1 (qty)

  3           LOAD_NAME                0 (price)
              LOAD_NAME                1 (qty)
              BINARY_OP                5 (*)
              STORE_NAME               2 (total)

  4           LOAD_NAME                3 (print)
              PUSH_NULL
              LOAD_NAME                2 (total)
              CALL                     1
              POP_TOP
              LOAD_CONST               1 (None)
              RETURN_VALUE


In [1]:
import opcode

code_string = "price = 10\nqty = 20\ntotal = price * qty\nprint(total)"

# 1. Compile the string into a code object
compiled_code = compile(code_string, '<string>', 'exec')

# 2. Extract the raw bytes, constants, and names
raw_bytes = list(compiled_code.co_code)
constants = compiled_code.co_consts
names = compiled_code.co_names

print("--- RAW ARRAYS ---")
print(f"co_code (raw bytes): {raw_bytes}")
print(f"co_consts (constants): {constants}")
print(f"co_names (variable names): {names}\n")

print("--- MANUAL BYTCODE DECODING ---")
# 3. Step through the byte array 2 bytes at a time (Opcode, Argument)
for i in range(0, len(raw_bytes), 2):
    op = raw_bytes[i]
    arg = raw_bytes[i+1]
    
    # Translate the integer op numeric value to its string name
    op_name = opcode.opname[op]
    
    # Contextualize what the argument points to based on the instruction
    context = ""
    if op_name in opcode.hasconst:
        context = f"-> value: {constants[arg]}"
    elif op_name in opcode.hasname:
        context = f"-> variable: '{names[arg]}'"
        
    print(f"Byte Offset {i:02d}: Opcode {op:<3} ({op_name:<12}) | Argument {arg:<3} {context}")

--- RAW ARRAYS ---
co_code (raw bytes): [128, 0, 94, 10, 116, 0, 94, 20, 116, 1, 93, 0, 93, 1, 44, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 116, 2, 93, 3, 33, 0, 93, 2, 52, 1, 0, 0, 0, 0, 0, 0, 31, 0, 82, 1, 35, 0]
co_consts (constants): (10, None)
co_names (variable names): ('price', 'qty', 'total', 'print')

--- MANUAL BYTCODE DECODING ---
Byte Offset 00: Opcode 128 (RESUME      ) | Argument 0   
Byte Offset 02: Opcode 94  (LOAD_SMALL_INT) | Argument 10  
Byte Offset 04: Opcode 116 (STORE_NAME  ) | Argument 0   
Byte Offset 06: Opcode 94  (LOAD_SMALL_INT) | Argument 20  
Byte Offset 08: Opcode 116 (STORE_NAME  ) | Argument 1   
Byte Offset 10: Opcode 93  (LOAD_NAME   ) | Argument 0   
Byte Offset 12: Opcode 93  (LOAD_NAME   ) | Argument 1   
Byte Offset 14: Opcode 44  (BINARY_OP   ) | Argument 5   
Byte Offset 16: Opcode 0   (CACHE       ) | Argument 0   
Byte Offset 18: Opcode 0   (CACHE       ) | Argument 0   
Byte Offset 20: Opcode 0   (CACHE       ) | Argument 0   
Byte Offset 22: Opcode